## 0. Importing PyTorch and setting up device-agnostic code

In [ ]:
import os
import random
import torch
import torch.nn as nn 

import torchvision
import torchvision.transforms as transforms 
from torchvision import datasets
from torch.utils.data import Dataset , DataLoader

import requests
import zipfile

from typing import Tuple, Dict, List

from pathlib import Path
from PIL import Image

import matplotlib.pyplot as plt
import numpy as np

torch.__version__

In [ ]:
# Setup device-agnostic code

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Device: ", device)

## 1. Get Data

In [ ]:
# setup path to a data folder 

data_path = Path("data/")

image_path = data_path / "pizza_steak_sushi"

# if the image folder doesn't exist then download it and prepare it......

if image_path.is_dir():
    print(f"{image_path} directory already exists... skipping download")

else:
    print(f"{image_path} does not exist, creating one....")
    image_path.mkdir(parents=True, exist_ok=True)

    # Download pizza_steak_sushi data

    with open( data_path / "pizza_steak_sushi.zip", "wb") as f:
        request = requests.get("https://github.com/mrdbourke/pytorch-deep-learning/raw/main/data/pizza_steak_sushi.zip")

        print("Download pizza , steak and sushi .....")

        f.write(request.content)


    # unzip pizza, steak, sushi data
    with zipfile.ZipFile(data_path / "pizza_steak_sushi.zip", "r") as zip_ref:
        print("Unzipping the pizza, steak and sushi data....")

        zip_ref.extractall(image_path)



## 2. Becoming one with the data (data preparation and data exploration)

In [ ]:
def walk_through_dir(dir_path):
     """
  Walks through dir_path returning its contents.
  Args:
    dir_path (str or pathlib.Path): target directory
  
  Returns:
    A print out of:
      number of subdirectories in dir_path
      number of images (files) in each subdirectory
      name of each subdirectory
  """

     for dirpath, dirnames, filenames in os.walk(dir_path):
          print(f"There are {len(dirnames)} directories and {len(filenames)} images in '{dirpath}. ")

In [ ]:
walk_through_dir(image_path)

In [ ]:
# setup train and testing paths 

train_dir = image_path / "train"

test_dir = image_path / "test"

train_dir, test_dir

## 2.1  Visualizing and Image 
Let's write some code to:
1. Get all of the images paths 
2. Get random image path's using `random.choice.`
3. Get the image class name using ` pathlib.path.parent.stem  `
4. Since we're working with images, let's open the image with Python's PIL
5. We'll then show the image and print metadata.


In [ ]:
# set the seed 

random.seed(42)

# Get all the image paths 
image_path_list  = list(image_path.glob("*/*/*.jpg"))

# pick a random image path 

random_image_path = random.choice(image_path_list)


# Get the image class name from the image path name (the image class is the name of the directory where all image is located.)

image_class = random_image_path.parent.stem

# Open Image 

img = Image.open(random_image_path)


# Print the metadata

print(f"Random Image Path: {random_image_path}")
print(f"Image Class: {image_class}")
print(f"Image height: {img.height}")
print(f"Image width: {img.width}")
img


## 2.2 Visualizing Image using MatPlotlib

In [ ]:
img_as_array = np.asanyarray(img)

plt.figure(figsize=(8, 7))
plt.imshow(img_as_array)
plt.title(f"Image class: {image_class} | Image shape: {img_as_array.shape} -> [height, width, color_channels]")
plt.axis(False);

## 3. Transforming Data
Now what if we wanted to load our image data into PyTorch?

Before we can use our image data with PyTorch we need to:

1. Turn it into tensors (numerical representations of our images).
2. Turn it into a torch.utils.data.Dataset and subsequently a torch.utils.data.DataLoader, we'll call these Dataset and DataLoader for short.


## 3.1 Transforming data with `torchvision.transforms`

In [ ]:
data_transform = transforms.Compose([
    transforms.Resize(size=(64, 64)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor()
])

print(f"Image Size Before transforms: {img.size}")
print(f"Image Size after transforms: {data_transform(img).shape}")
print(f"Image dtype after transforms: {data_transform(img).dtype}")

In [ ]:
def plot_transformed_images(image_paths, transform, n=3, seed=42):

    """Plots a series of random images from image_paths.

    Will open n image paths from image_paths, transform them
    with transform and plot them side by side.

    Args:
        image_paths (list): List of target image paths. 
        transform (PyTorch Transforms): Transforms to apply to images.
        n (int, optional): Number of images to plot. Defaults to 3.
        seed (int, optional): Random seed for the random generator. Defaults to 42.
    """

    random.seed(seed)

    random_image_paths = random.sample(image_paths, k=n)

    for img_path in random_image_paths:
        with Image.open(img_path) as f:
            fig, ax = plt.subplots(1,2)
            ax[0].imshow(f)
            ax[0].set_title(f"Original \nSize: {f.size}")
            ax[0].axis("off")

            # Transform and plot target image 

            transformed_image  = transform(f).permute(1, 2, 0)
            ax[1].imshow(transformed_image)
            ax[1].set_title(f"Transformed \nSize: {transformed_image.shape}")
            ax[1].axis("off")

            fig.suptitle(f"Class: {img_path.parent.stem}", fontsize=16)


plot_transformed_images(image_paths=image_path_list,
                        transform=data_transform,
                        n=3,
                        seed=42)


## 4. option 1: Loading image data using `ImageFolder`

We can load image classification data using `torchvision.datasets.ImageFolder.`

In [ ]:
# use ImageFolder to create dataset(s)

train_data = datasets.ImageFolder(root=train_dir,
                                  transform=data_transform,
                                  target_transform=None)

test_data = datasets.ImageFolder(root=test_dir,
                                  transform=data_transform,
                                  target_transform=None)

train_data, test_data

In [ ]:
# Get class names as list 

class_names = train_data.classes
class_names

In [ ]:
# Get class name as dict 

class_dict = train_data.class_to_idx

class_dict

In [ ]:
# check the length of our datasets 

len(train_data), len(test_data)

In [ ]:
train_data.samples[1]

In [ ]:
# Index on the train_data Dataset to get a single image and lable 


random.seed(42)

random_idx = random.randint(0, len(train_data) -1 )
img, label = train_data[random_idx]

print(f"Image tensor:\n{img}")
print(f"Image shape: {img.shape}")
print(f"Image datatype: {img.dtype}")
print(f"Image label: {label}")
print(f"Label datatype: {type(label)}")


In [ ]:
# Rearrange the order dimensions 

img_permute = img.permute(1, 2, 0)

print(f"Original shape: {img.shape} -> [color_channels, height, width]")
print(f"Image permute shape: {img_permute.shape} -> [height, width, color_channels]")


plt.figure(figsize=(9, 7))
plt.imshow(img_permute)
plt.axis(False)

plt.title(class_names[label], fontsize= 16)

## 4.1 Turn loaded images into DataLoader's

In [ ]:
# Turn train and test Datasets into DataLoaders

train_dataloader = DataLoader(dataset=train_data, 
                              batch_size=1, # how many samples per batch?
                              num_workers=5, # how many subprocesses to use for data loading? (higher = more)
                              shuffle=True) # shuffle the data?

test_dataloader = DataLoader(dataset=test_data, 
                             batch_size=1, 
                             num_workers=5, 
                             shuffle=False) # don't usually need to shuffle testing data

train_dataloader, test_dataloader

In [ ]:
im, lab = next(iter(train_dataloader))

# Batch size will now be 1, try changing the batch_size parameter above and see what happens
print(f"Image shape: {im.shape} -> [batch_size, color_channels, height, width]")
print(f"Label shape: {lab.shape}")

## 5 option 2: Loading Image Data with a custom `Dataset`

1.  Want to be able to load images from file 
2. want to be able to get class names from the Dataset 
3. want to be able to get  classes as dictionary from the Dataset 

Pros:
* Can create a `Dataset` out of almost anything 
* Not limited to PyTorch pre-built `Dataset` functions 


Cons:
* Even though you could create `Dataset` out of almost anything, it doesn't mean it will work....
* Using a custom `Dataset` often results in us writing more code, which could be prone to errors or performance issues

In [ ]:
# Instance of torchvision.datasets.ImageFolder()

train_data.classes, train_data.class_to_idx

### 5.1 Creating a helper function to get class names 

1. Get the class names using os.scandir() to traverse a target directory (ideally the directory is in standard image classification format).

2. Raise an error if the class names aren't found (if this happens, there might be something wrong with the directory structure).

3. Turn the class names into a dictionary of numerical labels, one for each class.

In [ ]:
# setup path for target directory


target_directory = train_dir

print(f"Target dir : {target_directory}")

# Get the class name from the target directory 

class_name_found = sorted([entry.name for entry in list(os.scandir(target_directory))])

class_name_found

In [ ]:
def find_classes(directory: str) -> Tuple[list[str], Dict[str, int]]:
    """Finds the class folder names in a target directory.
    
    Assumes target directory is in standard image classification format.

    Args:
        directory (str): target directory to load classnames from.

    Returns:
        Tuple[List[str], Dict[str, int]]: (list_of_class_names, dict(class_name: idx...))
    
    Example:
        find_classes("food_images/train")
        >>> (["class_1", "class_2"], {"class_1": 0, ...})
    """

     # 1. Get the class names by scanning the target directory

    classes = sorted(entry.name for entry in os.scandir(directory) if entry.is_dir())

    if not classes:
        raise FileExistsError(f"Couldn't find any classes in {directory}...")

    # 3. Create a dictionary of index labels (computers prefer numerical rather than string labels)

    class_to_idx = {class_name: i for i , class_name in enumerate(classes)}

    return classes, class_to_idx

In [ ]:
find_classes(target_directory)


## 5.2 Create a custom Dataset to replicate ImageFolder

* Subclass torch.utils.data.Dataset.


* Initialize our subclass with a targ_dir parameter (the target data directory) and transform parameter (so we have the option to transform our data if needed).


* Create several attributes for paths (the paths of our target images), transform (the transforms we might like to use, this can be None), classes and class_to_idx (from our find_classes() function).


* Create a function to load images from file and return them, this could be using PIL or torchvision.io (for input/output of vision data).


* Overwrite the __len__ method of torch.utils.data.Dataset to return the number of samples in the Dataset, this is recommended but not required. This is so you can call len(Dataset).


* Overwrite the __getitem__ method of torch.utils.data.Dataset to return a single sample from the Dataset, this is required.

In [ ]:
# Write a custom dataset class (inherits from torch.utils.data.Dataset)
from torch.utils.data import Dataset

# 1. Subclass torch.utils.data.Dataset
class ImageFolderCustom(Dataset):
    
    # 2. Initialize with a targ_dir and transform (optional) parameter
    def __init__(self, targ_dir: str, transform=None) -> None:
        
        # 3. Create class attributes
        # Get all image paths
        self.paths = list(Path(targ_dir).glob("*/*.jpg")) # note: you'd have to update this if you've got .png's or .jpeg's
        # Setup transforms
        self.transform = transform
        # Create classes and class_to_idx attributes
        self.classes, self.class_to_idx = find_classes(targ_dir)

    # 4. Make function to load images
    def load_image(self, index: int) -> Image.Image:
        "Opens an image via a path and returns it."
        image_path = self.paths[index]
        return Image.open(image_path) 
    
    # 5. Overwrite the __len__() method (optional but recommended for subclasses of torch.utils.data.Dataset)
    def __len__(self) -> int:
        "Returns the total number of samples."
        return len(self.paths)
    
    # 6. Overwrite the __getitem__() method (required for subclasses of torch.utils.data.Dataset)
    def __getitem__(self, index: int) -> Tuple[torch.Tensor, int]:
        "Returns one sample of data, data and label (X, y)."
        img = self.load_image(index)
        class_name  = self.paths[index].parent.name # expects path in data_folder/class_name/image.jpeg
        class_idx = self.class_to_idx[class_name]

        # Transform if necessary
        if self.transform:
            return self.transform(img), class_idx # return data, label (X, y)
        else:
            return img, class_idx # return data, label (X, y)

In [ ]:
# Augment train data
train_transforms = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor()
])

# Don't augment test data, only reshape
test_transforms = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor()
])

In [ ]:
train_data_custom = ImageFolderCustom(targ_dir=train_dir, 
                                      transform=train_transforms)
test_data_custom = ImageFolderCustom(targ_dir=test_dir, 
                                     transform=test_transforms)
train_data_custom, test_data_custom

In [ ]:
len(train_data_custom), len(test_data_custom)

In [ ]:
train_data_custom.classes

In [ ]:
train_data_custom.class_to_idx

In [ ]:
# Check for equality amongst our custom Dataset and ImageFolder Dataset
print((len(train_data_custom) == len(train_data)) & (len(test_data_custom) == len(test_data)))
print(train_data_custom.classes == train_data.classes)
print(train_data_custom.class_to_idx == train_data.class_to_idx)

## 5.3 Create a function to display random images

#### 1. Take in a Dataset and a number of other parameters such as classes (the names of our target classes), the number of images to display (n) and a random seed.

#### 2. To prevent the display getting out of hand, we'll cap n at 10 images.

#### 3. Set the random seed for reproducible plots (if seed is set).

#### 4. Get a list of random sample indexes (we can use Python's random.sample() for this) to plot.

#### 5. Setup a matplotlib plot.

#### 6. Loop through the random sample indexes found in step 4 and plot them with matplotlib.

#### 7. Make sure the sample images are of shape HWC (height, width, color channels) so we can plot them.

In [ ]:
# 1. Create a function to take in a dataset

def display_random_images(dataset: torch.utils.data.Dataset, 
                          classes: List[str] = None,
                          n: int = 10,
                          display_shape: bool = True,
                          seed: int = None):
    # 2. Adjust display if n is too high

    if n > 10:
        n =10
        display_shape = False

        print(f"For display purposes, n shouldn't be larger than 10, setting to 10 and removing shape display.")

    # 3. Set the seed 

    if seed:
        random.seed(seed)

    # 4. Get random sample indexes

    random_samples = random.sample(range(len(dataset)), k=n)

    # 5. Setup plot
    plt.figure(figsize=(16, 8))

    # 6. Loop through random indexes and plot them with matplotlib ....
    for i, targ_sample in enumerate(random_samples):
        targ_image , targ_label = dataset[targ_sample][0], dataset[targ_sample][1]


    # 7. Adjust image tensor shape for plotting: [color_channels, height, width] -> [color_channels, height, width]
        targ_image_adjust = targ_image.permute(1, 2, 0)
        # Plot adjusted samples
        plt.subplot(1, n, i+1)
        plt.imshow(targ_image_adjust)
        plt.axis("off")
        if classes:
            title = f"class: {classes[targ_label]}"
            if display_shape:
                title = title + f"\nshape: 6. Other forms of transforms (data augmentation){targ_image_adjust.shape}"
        plt.title(title)

    

In [ ]:
display_random_images(train_data_custom,
                      classes=train_data_custom.classes,
                      n=5,
                      seed=42)

In [ ]:
display_random_images(train_data,
                      classes=train_data.classes,
                      n=4,
                      seed=42)

## 5.4 Turn custom loaded images into DataLoader's

In [ ]:
train_dataloader_custom = DataLoader(dataset=train_data_custom, # use custom created train Dataset
                                     batch_size=1, # how many samples per batch?
                                     num_workers=0, # how many subprocesses to use for data loading? (higher = more)
                                     shuffle=True) # shuffle the data?

test_dataloader_custom = DataLoader(dataset=test_data_custom, # use custom created test Dataset
                                    batch_size=1, 
                                    num_workers=0, 
                                    shuffle=False) # don't usually need to shuffle testing data

train_dataloader_custom, test_dataloader_custom

In [ ]:
# Get image and label from custom DataLoader
img_custom, label_custom = next(iter(train_dataloader_custom))

# Batch size will now be 1, try changing the batch_size parameter above and see what happens
print(f"Image shape: {img_custom.shape} -> [batch_size, color_channels, height, width]")
print(f"Label shape: {label_custom.shape}")

##  6. Other forms of transforms (data augmentation)